# Cognito 2LO 인증, OpenAPI Target 및 Strands를 연동한 AgentCore Gateway

## 개요

최신 AI 에이전트 시스템이 MCP 서버를 통해 다운스트림 API에 접근하려면 안전한 인증 및 권한 부여 패턴이 필요합니다. 이 튜토리얼에서는 인바운드 인증과 아웃바운드 API 호출에 Cognito OAuth2 클라이언트 자격 증명 흐름(2LO)을 사용하는 AgentCore Gateway 기반의 종합적인 솔루션을 살펴봅니다.

이 솔루션에서 다루는 핵심 과제는 에이전트가 도구를 호출하고 해당 도구가 다시 다운스트림 API를 호출하는 멀티홉 워크플로에서 안전하게 토큰을 교환하고 자격 증명을 전파하는 것입니다. 조직은 MCP 서버에서 다운스트림 API로 범위가 축소된 토큰과 최소 권한 자격 증명을 보내기 위해 토큰을 교환해야 합니다. 이를 위해 인바운드 JWT 토큰에서 주체 자격 증명과 메타데이터를 추출하고, 호출자의 자격 증명을 기반으로 세분화된 접근 제어를 수행하며, 특정 다운스트림 API 호출에 적절한 범위의 자격 증명을 얻도록 토큰을 동적으로 교환해야 합니다.

이 솔루션은 각 홉이 별도의 범위 지정 토큰을 받는 안전한 대리 실행 패턴, 워크플로 전체에서 사용자 자격 증명을 유지하는 JWT 기반 실행 컨텍스트 전파, 적절한 데이터 분리를 위한 테넌트 격리, 규정 준수를 위한 명확한 감사 추적, 그리고 MCP 스키마를 그대로 유지하면서 인증을 별도로 처리하는 분리된 보안 구조를 지원합니다.

### 튜토리얼 세부 정보

| 정보                  | 세부 정보                                                 |
|:---------------------|:----------------------------------------------------------|
| 튜토리얼 유형         | 대화형                                                     |
| AgentCore 구성 요소  | AgentCore Gateway                                         |
| Gateway Target 유형  | OpenAPI                                                   |
| 인바운드 인증        | Cognito OAuth2 (Client Credentials)                       |
| 아웃바운드 인증      | Cognito OAuth2 (Client Credentials)                       |
| 튜토리얼 구성 요소   | AgentCore Gateway 생성 및 호출                            |
| 튜토리얼 분야        | 여러 분야에 공통 적용                                     |
| 예제 난이도          | 중급                                                        |
| 사용 SDK             | boto3, strands-agents                                     |

### 튜토리얼 아키텍처

![아키텍처 다이어그램](images/14-token-exchange-at-request-interceptor.png)

---

이 아키텍처는 다음을 보여 줍니다.

1. **클라이언트**: Cognito OAuth2 토큰으로 요청을 시작합니다.
2. **AgentCore Gateway**: 토큰 교환을 위해 인터셉터를 거쳐 클라이언트 요청을 라우팅합니다.
3. **Gateway 인터셉터**: 토큰을 검증하고 다운스트림 자격 증명으로 교환합니다.
4. **OpenAPI Target**: Cognito OAuth2 토큰과 함께 처리된 요청을 수신합니다.
5. **Strands 에이전트**: 인증된 Gateway와의 연동을 보여 줍니다.

이 Notebook에서는 다음 리소스를 생성합니다.
- Cognito 2LO 인증을 사용하는 AgentCore Gateway
- API 키 아웃바운드 인증을 사용하는 OpenAPI Target
- 스트리밍 가능한 HTTP 전송을 사용하는 Strands 에이전트 연동
- 여러 번 실행할 수 있도록 타임스탬프가 포함된 모든 리소스

## 1단계: 필수 패키지 설치

In [ ]:
# 필수 패키지 설치
!pip install --upgrade pip
!pip install boto3 requests
!pip install strands-agents

In [ ]:
import boto3
import json
import time
from datetime import datetime
import requests
from botocore.exceptions import ClientError

# 고유한 리소스 이름에 사용할 타임스탬프 생성
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
print(f"Timestamp: {timestamp}")

# AWS 클라이언트 초기화
session = boto3.Session()
region = session.region_name or "us-east-1"
print(f"Region: {region}")

cognito_client = boto3.client("cognito-idp", region_name=region)
agentcore_client = boto3.client("bedrock-agentcore-control", region_name=region)
iam_client = boto3.client("iam", region_name=region)

## 2단계: Cognito Resource Server 및 App Client 생성

In [ ]:
# Cognito User Pool 생성
user_pool_name = f"agentcore-pool-{timestamp}"

user_pool_response = cognito_client.create_user_pool(
    PoolName=user_pool_name,
    Policies={
        "PasswordPolicy": {
            "MinimumLength": 8,
            "RequireUppercase": False,
            "RequireLowercase": False,
            "RequireNumbers": False,
            "RequireSymbols": False,
        }
    },
)

user_pool_id = user_pool_response["UserPool"]["Id"]
print(f"User Pool ID: {user_pool_id}")

# V3_0 지원을 위해 User Pool을 확인하고 Essentials 티어로 업그레이드
try:
    pool_details = cognito_client.describe_user_pool(UserPoolId=user_pool_id)
    current_tier = pool_details["UserPool"].get("UserPoolTier", "Lite")
    print(f"Current User Pool tier: {current_tier}")

    if current_tier == "Lite":
        print("Upgrading User Pool to Essentials tier for V3_0 Pre Token Generation support...")
        cognito_client.update_user_pool(UserPoolId=user_pool_id, UserPoolTier="Essentials")
        print("User Pool upgraded to Essentials tier")
    else:
        print(f"User Pool is already on {current_tier} tier - V3_0 support available")
except Exception as e:
    print(f"Note: Could not check/upgrade User Pool tier: {e}")
    print("Please ensure User Pool is on Essentials or Plus tier for V3_0 Pre Token Generation")

## 3단계: Pre Token Generation Lambda 트리거 생성

In [ ]:
# Pre Token Generation Lambda용 IAM 역할 생성
token_lambda_role_name = f"PreTokenLambdaRole-{timestamp}"

token_lambda_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

token_lambda_role_response = iam_client.create_role(
    RoleName=token_lambda_role_name,
    AssumeRolePolicyDocument=json.dumps(token_lambda_trust_policy),
    Description="IAM role for Pre Token Generation Lambda",
)

# 기본 Lambda 실행 정책 연결
iam_client.attach_role_policy(
    RoleName=token_lambda_role_name,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
)

token_lambda_role_arn = token_lambda_role_response["Role"]["Arn"]
print(f"Pre Token Lambda IAM role created: {token_lambda_role_arn}")

# 역할을 사용할 수 있을 때까지 대기
time.sleep(10)

In [ ]:
# Pre Token Generation Lambda 함수 코드
import zipfile
import io

token_lambda_code = """
import json
import boto3
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)

def lambda_handler(event, context):
    logger.info("DEBUG - Pre Token Generation Lambda triggered")
    logger.info(f"DEBUG - Event: {json.dumps(event, default=str)}")
    logger.info(f"DEBUG - Trigger Source: {event.get('triggerSource', 'Unknown')}")
    
    # ID 및 access token 사용자 지정에 모두 V3_0 형식 사용
    event['response']['claimsAndScopeOverrideDetails'] = {
        'idTokenGeneration': {
            'claimsToAddOrOverride': {
                'custom:role': 'agentcore_user',
                'custom:permissions': 'read,write',
                'custom:tenant': 'default',
                'custom:api_access': 'enabled'
            },
            'claimsToSuppress': []
        },
        'accessTokenGeneration': {
            'claimsToAddOrOverride': {
                'custom:role': 'agentcore_user',
                'custom:permissions': 'read,write',
                'custom:tenant': 'default',
                'custom:api_access': 'enabled'
            },
            'claimsToSuppress': [],
            'scopesToAdd': [],
            'scopesToSuppress': []
        }
    }
    
    logger.info("DEBUG - Custom claims added to both ID and access tokens")
    logger.info(f"DEBUG - Response: {json.dumps(event['response'], default=str)}")
    
    return event
"""

# Pre Token Lambda용 ZIP 파일 생성
token_zip_buffer = io.BytesIO()
with zipfile.ZipFile(token_zip_buffer, "w", zipfile.ZIP_DEFLATED) as zip_file:
    zip_file.writestr("lambda_function.py", token_lambda_code)

token_zip_buffer.seek(0)

# Pre Token Generation Lambda 함수 생성
token_lambda_function_name = f"pre-token-generation-{timestamp}"

lambda_client = boto3.client("lambda", region_name=region)
token_lambda_response = lambda_client.create_function(
    FunctionName=token_lambda_function_name,
    Runtime="python3.13",
    Role=token_lambda_role_arn,
    Handler="lambda_function.lambda_handler",
    Code={"ZipFile": token_zip_buffer.read()},
    Description="Pre Token Generation Lambda for Cognito User Pool",
)

token_lambda_arn = token_lambda_response["FunctionArn"]
print(f"Pre Token Generation Lambda created: {token_lambda_arn}")

# Cognito가 호출할 수 있도록 Lambda 권한 추가
sts_client = boto3.client("sts", region_name=region)
account_id = sts_client.get_caller_identity()["Account"]

lambda_client.add_permission(
    FunctionName=token_lambda_function_name,
    StatementId="cognito-trigger-permission",
    Action="lambda:InvokeFunction",
    Principal="cognito-idp.amazonaws.com",
    SourceArn=f"arn:aws:cognito-idp:{region}:{account_id}:userpool/{user_pool_id}",
)

print("Lambda permission added for Cognito trigger")

In [ ]:
# Pre Token Generation 트리거로 User Pool 업데이트
cognito_client.update_user_pool(
    UserPoolId=user_pool_id,
    LambdaConfig={
        "PreTokenGeneration": token_lambda_arn,
        "PreTokenGenerationConfig": {
            "LambdaVersion": "V3_0",
            "LambdaArn": token_lambda_arn,
        },
    },
)

print("User Pool updated with Pre Token Generation trigger")
print("Custom claims will be added: role, permissions, tenant, api_access")
print("NOTE: Pre Token Generation Lambda only triggers for user authentication flows, NOT client_credentials flow")
print("For client_credentials flow, scopes are granted directly without Lambda trigger")

In [ ]:
# 2LO용 Resource Server 생성
resource_server_response = cognito_client.create_resource_server(
    UserPoolId=user_pool_id,
    Identifier=f"agentcore-api-{timestamp}",
    Name=f"AgentCore API {timestamp}",
    Scopes=[
        {"ScopeName": "read", "ScopeDescription": "Read access to AgentCore Gateway"},
        {"ScopeName": "write", "ScopeDescription": "Write access to AgentCore Gateway"},
    ],
)

resource_server_id = resource_server_response["ResourceServer"]["Identifier"]
print(f"Resource Server ID: {resource_server_id}")

In [ ]:
# 2LO용 App Client 생성(Client Credentials)
app_client_response = cognito_client.create_user_pool_client(
    UserPoolId=user_pool_id,
    ClientName=f"agentcore-client-{timestamp}",
    GenerateSecret=True,
    AllowedOAuthFlows=["client_credentials"],
    AllowedOAuthFlowsUserPoolClient=True,
    AllowedOAuthScopes=[f"{resource_server_id}/read", f"{resource_server_id}/write"],
    SupportedIdentityProviders=["COGNITO"],
)

client_id = app_client_response["UserPoolClient"]["ClientId"]
print(f"Client ID: {client_id}")

In [ ]:
# 클라이언트 보안 암호 가져오기
client_details = cognito_client.describe_user_pool_client(UserPoolId=user_pool_id, ClientId=client_id)

client_secret = client_details["UserPoolClient"]["ClientSecret"]
print(f"Client Secret: {client_secret[:10]}...")

In [ ]:
# User Pool 도메인 생성
domain_name = f"agentcore-{timestamp}"

try:
    domain_response = cognito_client.create_user_pool_domain(Domain=domain_name, UserPoolId=user_pool_id)
    print(f"Domain created: {domain_name}")
except ClientError as e:
    if "Domain already exists" in str(e):
        print(f"Domain {domain_name} already exists, continuing...")
    else:
        raise e

# Lambda 환경에 사용할 cognito_domain 설정(전체 도메인 URL)
cognito_domain = f"{domain_name}.auth.{region}.amazoncognito.com"

# 토큰 엔드포인트 구성
token_endpoint = f"https://{domain_name}.auth.{region}.amazoncognito.com/oauth2/token"
print(f"Token Endpoint: {token_endpoint}")

## 4단계: Gateway 인터셉터 Lambda 함수 생성

In [ ]:
import zipfile
import io

# 인터셉터 Lambda용 IAM 역할 생성
interceptor_role_name = f"InterceptorLambdaRole-{timestamp}"

interceptor_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

interceptor_role_response = iam_client.create_role(
    RoleName=interceptor_role_name,
    AssumeRolePolicyDocument=json.dumps(interceptor_trust_policy),
    Description="IAM role for Gateway Interceptor Lambda",
)

# 기본 Lambda 실행 정책 연결
iam_client.attach_role_policy(
    RoleName=interceptor_role_name,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
)

interceptor_role_arn = interceptor_role_response["Role"]["Arn"]
print(f"Interceptor Lambda IAM role created: {interceptor_role_arn}")

# 역할을 사용할 수 있을 때까지 대기
time.sleep(10)

In [ ]:
# Gateway 인터셉터용 Lambda 함수 코드
interceptor_code = """
import json
import boto3
import logging
import os
import urllib3
import base64

logger = logging.getLogger()
logger.setLevel(logging.INFO)

def lambda_handler(event, context):
    logger.info(f"Interceptor received event: {json.dumps(event, default=str)}")
    # MCP 구조에서 gateway request 추출
    mcp_data = event.get('mcp', {})
    gateway_request = mcp_data.get('gatewayRequest', {})
    headers = gateway_request.get('headers', {})
    body = gateway_request.get('body', {})
    
    logger.info(f"Headers: {headers}")
    logger.info(f"Body keys: {list(body.keys())}")
    
    # Token 교환용 authorization token 추출
    auth_header = headers.get('authorization', '') or headers.get('Authorization', '')
    logger.info(f"Auth header present: {bool(auth_header)}")
    logger.info(f"DEBUG - Incoming access token: {auth_header}")

    enhanced_token = auth_header
    
    # Cognito를 호출해 새 token 가져오기(Pre Token Generation Lambda 자동 실행)
    if auth_header:
        try:
            logger.info("Calling Cognito token endpoint for token exchange")
            
            # 환경 변수 가져오기
            client_id = os.environ.get('CLIENT_ID')
            client_secret = os.environ.get('CLIENT_SECRET')
            cognito_domain = os.environ.get('COGNITO_DOMAIN')
            resource_server_id = os.environ.get('RESOURCE_SERVER_ID')
            
            if not all([client_id, client_secret, cognito_domain, resource_server_id]):
                logger.error("Missing required environment variables")
                return
            
            # Token request 준비
            http = urllib3.PoolManager()
            token_url = f"https://{cognito_domain}/oauth2/token"
            
            # Basic 인증 header
            auth_string = f"{client_id}:{client_secret}"
            auth_bytes = auth_string.encode('ascii')
            auth_b64 = base64.b64encode(auth_bytes).decode('ascii')
            
            headers = {
                'Authorization': f'Basic {auth_b64}',
                'Content-Type': 'application/x-www-form-urlencoded'
            }
            
            cognito_body = f"grant_type=client_credentials&scope={resource_server_id}/read {resource_server_id}/write"
            
            response = http.request('POST', token_url, headers=headers, body=cognito_body)
            
            if response.status == 200:
                token_data = json.loads(response.data.decode('utf-8'))
                if 'access_token' in token_data:
                    enhanced_token = f"Bearer {token_data['access_token']}"
                    logger.info("Successfully obtained enhanced token from Cognito")
                    logger.info(f"DEBUG - Decorated access token: {enhanced_token}")
            else:
                logger.error(f"Token request failed with status {response.status}")
        except Exception as e:
            logger.error(f"Error getting enhanced token from Cognito: {str(e)}")

    
    # Request body를 처리하고 교환된 자격 증명 추가
    if "params" in body and "arguments" in body["params"]:
        # Argument에 향상된 authorization token 추가
        body["params"]["arguments"]["Authorization"] = enhanced_token
        logger.info("Added enhanced token to request arguments")
    
    # 변환된 요청 반환
    response = {
        "interceptorOutputVersion": "1.0",
        "mcp": {
            "transformedGatewayRequest": {
                "headers": {
                    "Accept": "application/json",
                    "Content-Type": "application/json"
                },
                "body": body
            }
        }
    }
    
    logger.info("DEBUG - Returning transformed request")
    logger.info(f"DEBUG - Transformed request: {json.dumps(response, default=str)}")
    
    return response
"""

# Lambda용 ZIP 파일 생성
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zip_file:
    zip_file.writestr("lambda_function.py", interceptor_code)

zip_buffer.seek(0)

# Lambda 함수 생성
interceptor_function_name = f"gateway-interceptor-{timestamp}"

lambda_client = boto3.client("lambda", region_name=region)
interceptor_response = lambda_client.create_function(
    FunctionName=interceptor_function_name,
    Runtime="python3.13",
    Role=interceptor_role_arn,
    Handler="lambda_function.lambda_handler",
    Code={"ZipFile": zip_buffer.read()},
    Environment={
        "Variables": {
            "CLIENT_ID": client_id,
            "CLIENT_SECRET": client_secret,
            "COGNITO_DOMAIN": cognito_domain,
            "RESOURCE_SERVER_ID": resource_server_id,
        }
    },
    Description="Gateway Interceptor for AgentCore Gateway",
)

interceptor_lambda_arn = interceptor_response["FunctionArn"]
print(f"Interceptor Lambda function created: {interceptor_lambda_arn}")

## 5단계: AgentCore Gateway용 IAM 역할 생성

In [ ]:
# AgentCore Gateway용 IAM 역할 생성
role_name = f"AgentCoreGatewayRole-{timestamp}"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

role_response = iam_client.create_role(
    RoleName=role_name,
    AssumeRolePolicyDocument=json.dumps(trust_policy),
    Description=f"IAM role for AgentCore Gateway {timestamp}",
)

# 인터셉터용 Lambda 호출 권한 추가
iam_client.put_role_policy(
    RoleName=role_name,
    PolicyName="LambdaInvokePolicy",
    PolicyDocument=json.dumps(
        {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Effect": "Allow",
                    "Action": ["lambda:InvokeAsync", "lambda:InvokeFunction"],
                    "Resource": "*",
                }
            ],
        }
    ),
)

role_arn = role_response["Role"]["Arn"]

# BedrockAgentCoreFullAccess 정책 연결
iam_client.attach_role_policy(RoleName=role_name, PolicyArn="arn:aws:iam::aws:policy/BedrockAgentCoreFullAccess")

print(f"Role ARN: {role_arn}")
print("BedrockAgentCoreFullAccess policy attached")

# 역할을 사용할 수 있을 때까지 대기
time.sleep(10)

## 6단계: API 키 인증을 사용하는 OpenAPI 명세 생성

In [ ]:
# 대상 서비스의 OpenAPI 명세 정의
openapi_spec = {
    "openapi": "3.0.1",
    "info": {
        "title": f"Sample API {timestamp}",
        "version": "1.0.0",
        "description": "Sample API with OAuth2 authentication",
    },
    "servers": [
        {
            "url": "https://jsonplaceholder.typicode.com",
            "description": "Sample API server",
        }
    ],
    "components": {
        "securitySchemes": {
            "OAuth2": {
                "type": "oauth2",
                "flows": {
                    "clientCredentials": {
                        "tokenUrl": f"https://{cognito_domain}/oauth2/token",
                        "scopes": {
                            f"{resource_server_id}/read": "Read access to AgentCore Gateway",
                            f"{resource_server_id}/write": "Write access to AgentCore Gateway",
                        },
                    }
                },
            }
        },
        "schemas": {
            "Post": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer"},
                    "title": {"type": "string"},
                    "body": {"type": "string"},
                    "userId": {"type": "integer"},
                },
            }
        },
    },
    "security": [{"OAuth2": []}],
    "paths": {
        "/posts": {
            "post": {
                "summary": "Create a new post",
                "operationId": "createPost",
                "parameters": [
                    {
                        "in": "header",
                        "name": "Authorization",
                        "schema": {"type": "string"},
                        "required": True,
                        "description": "Bearer token for authentication",
                    }
                ],
                "responses": {
                    "200": {
                        "description": "List of posts",
                        "content": {
                            "application/json": {
                                "schema": {
                                    "type": "array",
                                    "items": {"$ref": "#/components/schemas/Post"},
                                }
                            }
                        },
                    }
                },
                "security": [{"OAuth2": []}],
                "x-amazon-apigateway-integration": {
                    "type": "mock",
                    "requestTemplates": {"application/json": '{"statusCode": 200}'},
                    "responses": {
                        "default": {
                            "statusCode": "200",
                            "responseTemplates": {
                                "application/json": '[{"id": 1, "title": "Sample Post", "body": "This is a sample post", "userId": 1}]'
                            },
                        }
                    },
                },
            }
        }
    },
}

print("OpenAPI specification created")
print(json.dumps(openapi_spec, indent=2)[:500] + "...")

## 7단계: OpenAPI 명세로 API Gateway 생성

In [ ]:
# OpenAPI 명세로 API Gateway 생성
apigateway_client = boto3.client("apigateway", region_name=region)

# 권한 부여자에 사용할 계정 ID 가져오기
sts_client = boto3.client("sts", region_name=region)
account_id = sts_client.get_caller_identity()["Account"]

# OpenAPI 명세에서 API 가져오기
api_response = apigateway_client.import_rest_api(body=json.dumps(openapi_spec))

api_id = api_response["id"]
api_name = api_response["name"]
print(f"API Gateway created: {api_id}")
print(f"API Name: {api_name}")

# Cognito 권한 부여자 생성
authorizer_response = apigateway_client.create_authorizer(
    restApiId=api_id,
    name=f"cognito-authorizer-{timestamp}",
    type="COGNITO_USER_POOLS",
    providerARNs=[f"arn:aws:cognito-idp:{region}:{account_id}:userpool/{user_pool_id}"],
    identitySource="method.request.header.Authorization",
)

authorizer_id = authorizer_response["id"]
print(f"Cognito authorizer created: {authorizer_id}")

api_key_response = apigateway_client.create_api_key(
    name=f"agentcore-api-key-{timestamp}",
    description="API key for AgentCore Gateway outbound auth",
    enabled=True,
)

api_key_id = api_key_response["id"]
api_key_value = api_key_response["value"]
print(f"API Key created: {api_key_id}")
print(f"API Key value: {api_key_value[:10]}...")

# API를 먼저 배포
deployment_response = apigateway_client.create_deployment(
    restApiId=api_id,
    stageName="prod",
    description=f"Production deployment - {timestamp}",
)

print("API deployed to prod stage")

# 배포 후 사용량 계획 생성
usage_plan_response = apigateway_client.create_usage_plan(
    name=f"agentcore-usage-plan-{timestamp}",
    description="Usage plan for AgentCore Gateway",
    apiStages=[{"apiId": api_id, "stage": "prod"}],
    throttle={"rateLimit": 1000, "burstLimit": 2000},
    quota={"limit": 10000, "period": "DAY"},
)

usage_plan_id = usage_plan_response["id"]
print(f"Usage plan created: {usage_plan_id}")

# API 키를 사용량 계획에 연결
apigateway_client.create_usage_plan_key(usagePlanId=usage_plan_id, keyId=api_key_id, keyType="API_KEY")

print("API key associated with usage plan")

# AgentCore Gateway용 API 키 자격 증명 공급자 생성
credential_provider_response = agentcore_client.create_api_key_credential_provider(
    name=f"api-key-1provider-{timestamp}", apiKey="dummy-api-key-12345"
)

print(f"Credential provider response: {credential_provider_response}")
credential_provider_arn = credential_provider_response["credentialProviderArn"]
print(f"API key credential provider created: {credential_provider_arn}")

# API Gateway URL 구성
api_gateway_url = f"https://{api_id}.execute-api.{region}.amazonaws.com/prod"
print(f"API Gateway URL: {api_gateway_url}")

## 7.5단계: 액세스 토큰으로 API Gateway 테스트

In [ ]:
# Cognito 액세스 토큰으로 API Gateway 엔드포인트 테스트
import base64

print(f"Testing API Gateway endpoint: {api_gateway_url}")
print(f"Using Cognito token endpoint: https://{cognito_domain}/oauth2/token")

# Cognito에서 액세스 토큰 가져오기
credentials = f"{client_id}:{client_secret}"
encoded_credentials = base64.b64encode(credentials.encode()).decode()

token_headers = {
    "Authorization": f"Basic {encoded_credentials}",
    "Content-Type": "application/x-www-form-urlencoded",
}

token_data = {
    "grant_type": "client_credentials",
    "scope": f"{resource_server_id}/read {resource_server_id}/write",
}

# 액세스 토큰 요청
token_response = requests.post(f"https://{cognito_domain}/oauth2/token", headers=token_headers, data=token_data)

if token_response.status_code == 200:
    token_info = token_response.json()
    access_token = token_info["access_token"]
    print("✅ Access token obtained successfully")
    print(f"Token type: {token_info.get('token_type', 'Bearer')}")
    print(f"Expires in: {token_info.get('expires_in', 'N/A')} seconds")

    # API Gateway 엔드포인트 테스트
    api_headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }

    print("\n🔄 Testing API Gateway endpoint...")
    api_response = requests.post(
        f"{api_gateway_url}/posts",
        headers=api_headers,
        json={"title": "Test Post", "body": "This is a test post", "userId": 1},
    )

    print(f"API Response Status: {api_response.status_code}")
    print(f"API Response Headers: {dict(api_response.headers)}")

    if api_response.status_code == 200:
        print("✅ API call successful!")
        print(f"Response: {api_response.text[:200]}...")
    else:
        print("❌ API call failed")
        print(f"Error: {api_response.text}")

else:
    print("❌ Failed to get access token")
    print(f"Status: {token_response.status_code}")
    print(f"Error: {token_response.text}")

## 8단계: AgentCore Gateway 생성

In [ ]:
# Cognito 2LO 인증 및 OpenAPI Target을 사용하는 AgentCore Gateway 생성
gateway_name = f"agentcore-gateway-{timestamp}"

try:
    gateway_response = agentcore_client.create_gateway(
        name=gateway_name,
        description=f"AgentCore Gateway with Cognito 2LO auth - {timestamp}",
        roleArn=role_arn,
        protocolType="MCP",
        protocolConfiguration={"mcp": {"supportedVersions": ["2025-03-26", "2025-06-18"]}},
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration",
                "allowedClients": [client_id],
                "allowedScopes": [
                    f"{resource_server_id}/read",
                    f"{resource_server_id}/write",
                ],
            }
        },
        interceptorConfigurations=[
            {
                "interceptor": {"lambda": {"arn": interceptor_lambda_arn}},
                "interceptionPoints": ["REQUEST"],
                "inputConfiguration": {"passRequestHeaders": True},
            }
        ],
    )

    print(f"Gateway response: {json.dumps(gateway_response, indent=2, default=str)}")

    # 가능한 여러 응답 구조 처리
    if "gateway" in gateway_response:
        gateway_id = gateway_response["gateway"]["gatewayId"]
        gateway_arn = gateway_response["gateway"]["gatewayArn"]
    elif "gatewayId" in gateway_response:
        gateway_id = gateway_response["gatewayId"]
        gateway_arn = gateway_response.get("gatewayArn", "N/A")
    else:
        gateway_id = "Unknown"
        gateway_arn = "Unknown"

    print("Gateway created successfully!")
    print(f"Gateway ID: {gateway_id}")
    print(f"Gateway ARN: {gateway_arn}")

except Exception as e:
    print(f"Error creating gateway: {str(e)}")
    print(f"Exception type: {type(e).__name__}")
    if hasattr(e, "response"):
        print(f"Error code: {e.response.get('Error', {}).get('Code', 'Unknown')}")
        print(f"Error message: {e.response.get('Error', {}).get('Message', 'Unknown')}")
        print(f"HTTP status: {e.response.get('ResponseMetadata', {}).get('HTTPStatusCode', 'Unknown')}")
    import traceback

    print(f"Full traceback:\n{traceback.format_exc()}")

## 9단계: API Gateway용 AgentCore Gateway Target 생성

In [ ]:
# Target을 생성하기 전에 Gateway가 준비될 때까지 대기
print("Waiting for gateway to be ready...")
while True:
    try:
        gateway_status = agentcore_client.get_gateway(gatewayIdentifier=gateway_id)
        if gateway_status.get("status") == "READY":
            gateway_url = gateway_status.get("gatewayUrl")
            print(f"Gateway is ready: {gateway_url}")
            break
        else:
            print(f"Gateway status: {gateway_status.get('status')}")
            time.sleep(10)
    except Exception as e:
        print(f"Error checking gateway status: {e}")
        time.sleep(10)

# API Gateway를 가리키는 AgentCore Gateway Target 생성
target_response = agentcore_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name=f"api-gateway-target-{timestamp}",
    targetConfiguration={
        "mcp": {
            "openApiSchema": {
                "inlinePayload": json.dumps(
                    {
                        **openapi_spec,
                        "servers": [
                            {
                                "url": api_gateway_url,
                                "description": "API Gateway endpoint",
                            }
                        ],
                    }
                )
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {
                    "providerArn": credential_provider_arn,
                    "credentialParameterName": "X-API-Key",
                    "credentialLocation": "HEADER",
                }
            },
        }
    ],
)

target_id = target_response["targetId"]
print(f"AgentCore Gateway target created: {target_id}")

# Target이 준비될 때까지 대기
print("Waiting for target to be ready...")
while True:
    try:
        target_status = agentcore_client.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
        if target_status.get("status") == "READY":
            print(f"Target is ready: {target_id}")
            break
        else:
            print(f"Target status: {target_status.get('status')}")
            time.sleep(10)
    except Exception as e:
        print(f"Error checking target status: {e}")
        time.sleep(10)

## 10단계: 2LO 인증 테스트

In [ ]:
# Cognito 2LO 토큰 생성 테스트
import base64

print(f"Testing token endpoint: {token_endpoint}")
print(f"Client ID: {client_id}")
print(f"Resource Server ID: {resource_server_id}")

# 클라이언트 자격 증명 준비
credentials = f"{client_id}:{client_secret}"
encoded_credentials = base64.b64encode(credentials.encode()).decode()

# 클라이언트 자격 증명 흐름으로 액세스 토큰 요청
token_request = {
    "grant_type": "client_credentials",
    "scope": f"{resource_server_id}/read {resource_server_id}/write",
}

headers = {
    "Authorization": f"Basic {encoded_credentials}",
    "Content-Type": "application/x-www-form-urlencoded",
}

print(f"Token request data: {token_request}")
print(f"Request headers: {dict(headers)}")

try:
    response = requests.post(token_endpoint, data=token_request, headers=headers)

    print(f"Response status: {response.status_code}")
    print(f"Response headers: {dict(response.headers)}")

    if response.status_code == 200:
        token_data = response.json()
        access_token = token_data["access_token"]
        print("Access token obtained successfully!")
        print(f"Token type: {token_data.get('token_type')}")
        print(f"Expires in: {token_data.get('expires_in')} seconds")
        print(f"Access token: {access_token}...")
    else:
        print(f"Token request failed: {response.status_code}")
        print(f"Response: {response.text}")

        # 도메인이 준비되었는지 확인
        print("\nChecking domain status...")
        try:
            domain_info = cognito_client.describe_user_pool_domain(Domain=domain_name)
            print(f"Domain status: {domain_info.get('DomainDescription', {}).get('Status', 'Unknown')}")
        except Exception as domain_e:
            print(f"Error checking domain: {domain_e}")

except Exception as e:
    print(f"Error testing authentication: {str(e)}")

## 11단계: 리소스 요약 및 정리 지침

In [ ]:
# 생성된 리소스 요약
print("=== CREATED RESOURCES SUMMARY ===")
print(f"Timestamp: {timestamp}")
print(f"Region: {region}")
print()
print("Cognito Resources:")
print(f"  User Pool ID: {user_pool_id}")
print(f"  User Pool Name: {user_pool_name}")
print(f"  Client ID: {client_id}")
print(f"  Resource Server ID: {resource_server_id}")
print(f"  Domain: {domain_name}")
print(f"  Token Endpoint: {token_endpoint}")
print()
print("API Gateway Resources:")
print(f"  API ID: {api_id}")
print(f"  API Name: {api_name}")
print(f"  API Gateway URL: {api_gateway_url}")
print(f"  API Key ID: {api_key_id}")
print(f"  Usage Plan ID: {usage_plan_id}")
print()
print("IAM Resources:")
print(f"  Gateway Role Name: {role_name}")
print(f"  Gateway Role ARN: {role_arn}")
print(f"  Interceptor Role Name: {interceptor_role_name}")
print(f"  Interceptor Lambda ARN: {interceptor_lambda_arn}")
print()
print("AgentCore Gateway:")
print(f"  Gateway ID: {gateway_id}")
print(f"  Gateway URL: {gateway_url}")
print(f"  Target ID: {target_id}")
print()
print("Authentication Flow:")
print("  Inbound: Cognito 2LO (Client Credentials)")
print(f"  Scopes: {resource_server_id}/read, {resource_server_id}/write")
print("  Outbound: API Key (X-API-Key header)")
print("  Target: API Gateway with OpenAPI spec")

## 12단계: Strands 에이전트 연동

이제 AgentCore Gateway 인증을 지원하는 Strands 에이전트를 연동해 보겠습니다.

In [ ]:
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent
import logging

# 로깅 구성
logging.getLogger("strands").setLevel(logging.INFO)
logging.basicConfig(format="%(levelname)s | %(name)s | %(message)s", handlers=[logging.StreamHandler()])


def create_streamable_http_transport():
    """OAuth token을 사용하는 transport를 생성합니다."""
    return streamablehttp_client(gateway_url, headers={"Authorization": f"Bearer {access_token}"})


client = MCPClient(create_streamable_http_transport)

# Bedrock 모델 생성
model = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    temperature=0.7,
)

print("✅ Strands agent configured with authentication")

In [ ]:
with client:
    # 사용 가능한 도구 나열
    tools = client.list_tools_sync()

    # 에이전트 생성
    agent = Agent(model=model, tools=tools)

    print(f"Tools loaded: {agent.tool_names}\n")

    # 테스트: 도구 목록 조회
    print("Test: List available tools")
    print("=" * 50)
    response = agent("Hi, can you list all tools available to you?")
    print(f"Agent response: {response}\n")

In [ ]:
with client:
    # 사용 가능한 도구 나열
    tools = client.list_tools_sync()

    # 에이전트 생성
    agent = Agent(model=model, tools=tools)

    print(f"Tools loaded: {agent.tool_names}\n")

    # 테스트: 도구 직접 호출
    print("Test: Direct tool call")
    print("=" * 50)

    # 사용 가능한 첫 번째 도구 이름 가져오기
    if tools:
        tool_name = tool_name = tools[0].tool_name
        print(f"Using tool: {tool_name}")

        result = client.call_tool_sync(
            tool_use_id="test-123",
            name=tool_name,
            arguments={"id": "Hello from Strands agent!"},
        )
        print(f"Tool result: {result['content'][0]['text']}")
    else:
        print("No tools available")

## 13단계: 리소스 정리

In [ ]:
# 정리 함수(모든 리소스를 삭제하려면 이 함수 실행)
def cleanup_resources():
    print("Starting cleanup...")

    try:
        # AgentCore Gateway Target 삭제
        agentcore_client.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
        print(f"Deleted gateway target: {target_id}")

        # Target 삭제가 완료될 때까지 대기
        print("Waiting for target deletion to complete...")
        while True:
            try:
                agentcore_client.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
                time.sleep(5)
            except Exception:
                print("Target deletion completed")
                break

    except Exception as e:
        print(f"Error deleting gateway target: {e}")

    try:
        # AgentCore Gateway 삭제
        agentcore_client.delete_gateway(gatewayIdentifier=gateway_id)
        print(f"Deleted gateway: {gateway_id}")
    except Exception as e:
        print(f"Error deleting gateway: {e}")

    try:
        # Lambda 함수 삭제
        lambda_client = boto3.client("lambda", region_name=region)
        lambda_client.delete_function(FunctionName=interceptor_function_name)
        print(f"Deleted interceptor Lambda: {interceptor_function_name}")

        lambda_client.delete_function(FunctionName=token_lambda_function_name)
        print(f"Deleted pre-token Lambda: {token_lambda_function_name}")
    except Exception as e:
        print(f"Error deleting Lambda functions: {e}")

    try:
        # API Gateway 삭제
        apigateway_client.delete_rest_api(restApiId=api_id)
        print(f"Deleted API Gateway: {api_id}")
    except Exception as e:
        print(f"Error deleting API Gateway: {e}")

    try:
        # Secrets Manager 보안 암호 삭제
        secretsmanager_client = boto3.client("secretsmanager", region_name=region)
        secretsmanager_client.delete_secret(SecretId=f"agentcore-api-key-{timestamp}", ForceDeleteWithoutRecovery=True)
        print(f"Deleted Secrets Manager secret: agentcore-api-key-{timestamp}")
    except Exception as e:
        print(f"Error deleting secret: {e}")

    try:
        # User Pool 도메인 삭제
        cognito_client.delete_user_pool_domain(Domain=domain_name, UserPoolId=user_pool_id)
        print(f"Deleted domain: {domain_name}")
        time.sleep(5)
    except Exception as e:
        print(f"Error deleting domain: {e}")

    try:
        # User Pool 삭제
        cognito_client.delete_user_pool(UserPoolId=user_pool_id)
        print(f"Deleted user pool: {user_pool_id}")
    except Exception as e:
        print(f"Error deleting user pool: {e}")

    try:
        # IAM 역할 정리 - 먼저 모든 정책 분리
        for role in [role_name, interceptor_role_name, token_lambda_role_name]:
            try:
                # 연결된 모든 관리형 정책을 나열하고 분리
                attached_policies = iam_client.list_attached_role_policies(RoleName=role)
                for policy in attached_policies["AttachedPolicies"]:
                    iam_client.detach_role_policy(RoleName=role, PolicyArn=policy["PolicyArn"])
                    print(f"Detached policy {policy['PolicyName']} from {role}")

                # 모든 인라인 정책을 나열하고 삭제
                inline_policies = iam_client.list_role_policies(RoleName=role)
                for policy_name in inline_policies["PolicyNames"]:
                    iam_client.delete_role_policy(RoleName=role, PolicyName=policy_name)
                    print(f"Deleted inline policy {policy_name} from {role}")

                # 이제 역할 삭제
                iam_client.delete_role(RoleName=role)
                print(f"Deleted IAM role: {role}")

            except Exception as e:
                print(f"Error with role {role}: {e}")
    except Exception as e:
        print(f"Error: {e}")
    print("Cleanup completed!")


# 정리를 실행하려면 아래 줄의 주석을 해제
cleanup_resources()